# Lesson 4

## Continuation of makemore - adding neural network to probability bigram to make model better

In [4]:
import sys
from pathlib import Path

# Add the parent directory (nnz2h) to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import pygrad

import math
import numpy as np
import matplotlib.pyplot as plt
import random
import torch # import torch for the usage of their tensors
import statistics
from pygrad import Scalar

# jupyter magic:
%matplotlib inline

Lets get back our model we had before, below will be the code that creates the previous bigram model we had

In [5]:
words = open('names.txt','r').read().splitlines() # set up word analysis

N = torch.zeros((27, 27), dtype=torch.int32) # create initial array

# set up a way to keep track of the characters

chars = sorted(list(set(''.join(words)))) # characters
stoi ={s:i+1 for i,s in enumerate(chars)} # enumerate gives us an iterator over the characters
stoi['.'] = 0 # Special 'start' or 'end' character (typically <#>)
itos = {i:s for s,i in stoi.items()}

# frequency of the bigram for the data set

for w in words: # need to make the start and end stand out with special tokens
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1 # stores frequency of the letter in the matrix


# turn it into a probability table

P = N.float()
P /= P.sum(1,keepdim=True) # makes a sum horizontally (sum of rows), normalizes rows

# when summing, broadcasting is very specific, keepdim is keeps the rows with the rows, while if there is none the rows would go with the columns


# first attempt at a model
g = torch.Generator().manual_seed(2147483647)

for i in range(100):

    out = []
    ix = 0 # start token
    while True:
        p = P[ix]
        ix = torch.multinomial(p,num_samples=1,replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            
            break

# introduction to the loss function

log_likelihood = 0.0
n = 0 ; 
for w in words: 
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1,ix2] 
        logprob = torch.log(prob) # turn to log since easier to work with log over super small number
        log_likelihood += logprob
        n += 1
        nll = -log_likelihood # nice loss function, higher the number the worse the predicition


We will now do the probability table, but with neural network

it will recieve a single input and go through several neurons and make a guess for the next character

first we need to create a training set of bigrams

In [6]:
from pygrad import Neuron # one instance of a single neuron which has weights and bias to calculated based on inputs
from pygrad import Layer # one instance of a layer which contains multiple neurons
from pygrad import MLP # one instance of a multi-layer perceptron which contains multiple layers of neurons


xs, ys = [] , [] # inputs ... targets


for w in words[:1]: # need to make the start and end stand out with special tokens
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

# lowercase tensor is returning data type integer while Tensor gives float32

xs = torch.tensor(xs)
ys = torch.tensor(ys)



Can't input placement of data into neural network directly, turn it into matrix

In [7]:
import torch.nn.functional as F

# Start of forward pass

xenc = F.one_hot(xs, num_classes=27).float() # takes xs and turns it into matrix feedable into neural network

# note xenc is int64 due to one_hot, have to cast it to float to fix that

In [8]:
W = torch.randn((27, 27)) # weights of a singular neuron
xenc @ W # matrix multiply in pytorch

tensor([[-0.1483, -0.9490, -0.6836,  0.9814,  1.7328,  1.0323,  1.2768, -0.6238,
         -0.5805, -0.0975, -0.4174,  0.7752, -1.1019,  0.4876,  1.2849, -1.1011,
          0.2243,  0.7915,  1.3560,  0.5806,  0.2886, -1.2973, -1.1187,  1.6560,
         -0.3372,  1.4357, -0.6647],
        [ 0.2134,  1.1678,  0.1439, -0.3244,  0.7495,  1.7317, -0.8489,  0.4464,
          0.3766,  0.8811,  1.0509, -1.0023, -1.2358,  1.5969,  0.8943,  0.2603,
          0.7009,  0.7694, -0.4275,  0.2603,  0.3220, -0.7088, -0.3888, -1.2312,
         -0.2720, -0.1312, -0.0713],
        [-0.2571,  1.7777,  0.2683, -0.9325,  0.6630,  0.4686, -0.3364,  0.5276,
          1.8443,  0.5850, -1.4527,  1.2166, -0.3897, -2.0362,  1.4882,  0.4073,
          0.3995,  0.2310,  2.2100, -1.2136, -0.5995,  0.3046,  1.2521,  0.8254,
         -0.0392, -0.6029, -0.5849],
        [-0.2571,  1.7777,  0.2683, -0.9325,  0.6630,  0.4686, -0.3364,  0.5276,
          1.8443,  0.5850, -1.4527,  1.2166, -0.3897, -2.0362,  1.4882,  0.4073

Output neuron interpretation, take log counts and exponentiate them

In [9]:
logits = (xenc @ W) # log-counts
counts = (xenc @ W).exp() # cequivalent to N
probs = counts / counts.sum(1,keepdims=True) # softmax counts-> probs (take outputs of neural layer and turns it to probability distributions)
probs

tensor([[0.0176, 0.0079, 0.0103, 0.0545, 0.1156, 0.0574, 0.0733, 0.0109, 0.0114,
         0.0185, 0.0135, 0.0444, 0.0068, 0.0333, 0.0738, 0.0068, 0.0256, 0.0451,
         0.0793, 0.0365, 0.0273, 0.0056, 0.0067, 0.1070, 0.0146, 0.0859, 0.0105],
        [0.0283, 0.0736, 0.0264, 0.0165, 0.0484, 0.1293, 0.0098, 0.0358, 0.0334,
         0.0552, 0.0655, 0.0084, 0.0067, 0.1130, 0.0560, 0.0297, 0.0461, 0.0494,
         0.0149, 0.0297, 0.0316, 0.0113, 0.0155, 0.0067, 0.0174, 0.0201, 0.0213],
        [0.0141, 0.1081, 0.0239, 0.0072, 0.0355, 0.0292, 0.0131, 0.0310, 0.1156,
         0.0328, 0.0043, 0.0617, 0.0124, 0.0024, 0.0809, 0.0275, 0.0272, 0.0230,
         0.1666, 0.0054, 0.0100, 0.0248, 0.0639, 0.0417, 0.0176, 0.0100, 0.0102],
        [0.0141, 0.1081, 0.0239, 0.0072, 0.0355, 0.0292, 0.0131, 0.0310, 0.1156,
         0.0328, 0.0043, 0.0617, 0.0124, 0.0024, 0.0809, 0.0275, 0.0272, 0.0230,
         0.1666, 0.0054, 0.0100, 0.0248, 0.0639, 0.0417, 0.0176, 0.0100, 0.0102],
        [0.1590, 0.0193,

Emma example

In [10]:
# emma example, 5 parts since 5 bigrams

nlls = torch.zeros(5)
for i in range(5):
    #i-th bigram
    x = xs[i].item() # input character idx
    y = ys[i].item() # label character idx
    print('----------------------')
    print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
    print('input to the neural net:',x)
    print('output probabilities from the neural net:', probs[i])
    print('label (actual next character):', y)
    p = probs[i,y]
    print('probability assigned by the net to the correct character', p.item()) # chance of our model predicting the 'right' character
    logp = torch.log(p)
    nll = -logp
    print('log likelihood', nll.item())
    nlls[i] = nll
print('=====================')
print('average negative log likelihood (loss) = ',nlls.mean().item())

----------------------
bigram example 1: .e (indexes 0,5)
input to the neural net: 0
output probabilities from the neural net: tensor([0.0176, 0.0079, 0.0103, 0.0545, 0.1156, 0.0574, 0.0733, 0.0109, 0.0114,
        0.0185, 0.0135, 0.0444, 0.0068, 0.0333, 0.0738, 0.0068, 0.0256, 0.0451,
        0.0793, 0.0365, 0.0273, 0.0056, 0.0067, 0.1070, 0.0146, 0.0859, 0.0105])
label (actual next character): 5
probability assigned by the net to the correct character 0.057364121079444885
log likelihood 2.8583362102508545
----------------------
bigram example 2: em (indexes 5,13)
input to the neural net: 5
output probabilities from the neural net: tensor([0.0283, 0.0736, 0.0264, 0.0165, 0.0484, 0.1293, 0.0098, 0.0358, 0.0334,
        0.0552, 0.0655, 0.0084, 0.0067, 0.1130, 0.0560, 0.0297, 0.0461, 0.0494,
        0.0149, 0.0297, 0.0316, 0.0113, 0.0155, 0.0067, 0.0174, 0.0201, 0.0213])
label (actual next character): 13
probability assigned by the net to the correct character 0.1130155548453331
log like

 Neural net optimization, always start with random weights and minimize the loss of the weight, lets develop it fully here to show how the model looks

In [11]:
# Initialize our input and output: xs, ys aka dataset

xs, ys = [] , [] # inputs ... targets


for w in words: # need to make the start and end stand out with special tokens
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

# lowercase tensor is returning data type integer while Tensor gives float32

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

# randomly generate weights, 27 neuron weights with 27 inputs (similar to P matrix), initializing our network

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27), generator = g, requires_grad = True)

for k in range(100):

    # forward pass

    xenc = F.one_hot(xs, num_classes = 27).float() # input to the network: one hot encoding
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1,keepdims = True) # probabilities for next character

    # loss calculation

    loss = - probs[torch.arange(num),ys].log().mean() # gets the probs of each character that is correct
    print(loss.item())
    # backward pass

    W.grad = None # set to zero the grad
    loss.backward()

    # update
    W.data += -10 * W.grad

# notice that this will get the same prediction as the bigram prob model, will converge to the same prob log loss before

3.758953809738159
3.6702587604522705
3.591153860092163
3.520017623901367
3.4557948112487793
3.397716522216797
3.3451638221740723
3.2975902557373047
3.2544844150543213
3.2153542041778564
3.1797327995300293
3.1471893787384033
3.117339611053467
3.0898516178131104
3.064443588256836
3.0408785343170166
3.0189590454101562
2.998518705368042
2.9794163703918457
2.9615297317504883
2.9447529315948486
2.92899227142334
2.9141643047332764
2.9001941680908203
2.887012243270874
2.874558448791504
2.862776279449463
2.851613759994507
2.841024398803711
2.830965518951416
2.8213980197906494
2.812286138534546
2.8035969734191895
2.7953009605407715
2.7873709201812744
2.7797818183898926
2.7725110054016113
2.7655375003814697
2.7588419914245605
2.7524070739746094
2.7462172508239746
2.7402572631835938
2.734513998031616
2.7289750576019287
2.7236287593841553
2.7184643745422363
2.713472366333008
2.708644390106201
2.7039713859558105
2.699446678161621
2.6950621604919434
2.6908111572265625
2.6866886615753174
2.68268775939

Lets make it more complicated to lower the loss function, the reason neural network is preferred is we can customize it to make it better, the area of most improvement will be in forward passes

In [12]:
# Initialize our input and output: xs, ys aka dataset

xs, ys = [] , [] # inputs ... targets


for w in words: # need to make the start and end stand out with special tokens
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

# lowercase tensor is returning data type integer while Tensor gives float32

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

# randomly generate weights, 27 neuron weights with 27 inputs (similar to P matrix), initializing our network

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27), generator = g, requires_grad = True)

for k in range(100):

    # forward pass

    xenc = F.one_hot(xs, num_classes = 27).float() # input to the network: one hot encoding
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1,keepdims = True) # probabilities for next character

    # loss calculation

    loss = - probs[torch.arange(num),ys].log().mean() + 0.01* (W**2).mean() # The W**2 is a regulation loss (like spring force), similar to the previous adding +1 to P
    print(loss.item())
    # backward pass

    W.grad = None # set to zero the grad
    loss.backward()

    # update
    W.data += -10 * W.grad

# notice that this will get the same prediction as the bigram prob model, will converge to the same prob log loss before

3.7686190605163574
3.6794421672821045
3.5999374389648438


3.528468370437622
3.463968515396118
3.4056591987609863
3.352914571762085
3.305180788040161
3.261941909790039
3.222700595855713
3.186985969543457
3.154364585876465
3.124450206756592
3.0969085693359375
3.0714569091796875
3.047856569290161
3.025909900665283
3.005449056625366
2.9863314628601074
2.968435287475586
2.951653242111206
2.935891628265381
2.92106556892395
2.907099962234497
2.8939261436462402
2.881481885910034
2.869711399078369
2.858562707901001
2.8479881286621094
2.8379459381103516
2.8283965587615967
2.819303512573242
2.8106343746185303
2.8023593425750732
2.7944512367248535
2.7868854999542236
2.7796385288238525
2.7726891040802
2.766019344329834
2.7596113681793213
2.753448247909546
2.747516393661499
2.7418017387390137
2.7362918853759766
2.730975389480591
2.7258412837982178
2.7208809852600098
2.7160844802856445
2.71144437789917
2.7069520950317383
2.7026009559631348
2.6983847618103027
2.6942965984344482
2.690331220626831
2.6864826679229736
2.6827468872070312
2.679119110107422
2.67559

In [18]:
# Let us sample from our network

for i in range(5):
    out = []
    ix = 0
    while True:

        xenc = F.one_hot(torch.tensor(([ix])), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts/counts.sum(1, keepdims=True)

        ix = torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()
        out.append(itos[ix])
        if ix== 0:
            break
    print(''.join(out))

kotallihalqrzw.
eie.
ke.
s.
s.


## The neural network now is on par with the bigram, different ways of reaching the same conclusion (aka same loss)

- bigram used one probability distribution and reached a cap with only smoothing allowing for customization, uses probability of the data set to determine whats next

- neural network randomizes what the weights are and calculates a loss (nll) to be squished down until reaching the actual output, creating similar probability to bigram

#### Neural will be preferred since it is heavily customizable and new methods can be put in to optimize the loss further and make it more accurate, while bigram has already peaked

Bonus: Lets try to make the neural net in python without torch

In [14]:


# # Initialize our input and output: xs, ys aka dataset

# total_loss = 0

# xs, ys = [] , [] # inputs ... targets


# for w in words: # need to make the start and end stand out with special tokens
#     chs = ['.'] + list(w) + ['.']
#     for ch1, ch2 in zip(chs,chs[1:]):
#         ix1 = stoi[ch1]
#         ix2 = stoi[ch2]
#         xs.append(ix1)
#         ys.append(ix2)

# # lowercase tensor is returning data type integer while Tensor gives float32

# xs = np.array(xs)
# ys = np.array(ys)
# num = xs.size

# # use MLP definition to make 27xx27 matrix

# W = MLP(num_i=27, num_os=[27],activation=None)


# for k in range(100):

    
#     losses = []
#     for ix1, ix2 in zip(xs, ys):
#         logits = W(ix1) # predict log-counts
#         counts = [math.e ** l for l in logits] # counts, equivalent to N
#         total_counts = sum(counts)
#         probs = [c / total_counts for c in counts] # probabilities for next character

#         # loss calculation

#         loss_i = -probs[ix2].log() + 0.01 * sum(p * p for p in W.parameters())/len(W.parameters())

#         print(loss_i)

#         total_loss += loss_i.value
#     # backward pass

#     for p in W.parameters():
#         p.grad = 0.0
#     loss_i.acc_grads()

#     # update
#     learning_rate = 0.03
#     for p in W.parameters():
#         p.value += -learning_rate * p.grad

# # This would take a very long time to compute

# # notice that this will get the same prediction as the bigram prob model, will converge to the same prob log loss before

In [15]:
# for i in range(10):
#     out = []
#     ix = 0
#     while True:

        
#         xenc = [1.0 if j == ix else 0.0 for j in range(27)] 
#         logits = W(xenc)
#         counts = [math.e ** l for l in logits] # counts, equivalent to N
#         total_counts = sum(counts)
#         probs = [c / total_counts for c in counts ]# probabilities for next character

#         prob_values = [p.value for p in probs]

#         ix = random.choices(range(27),weights=prob_values,k=1)[0]

#         out.append(itos[ix])
#         if ix == 0:
#             break
        
        
#     print(''.join(out))